# Power BI Dataset Generation

In [1]:
import pandas as pd

orders    = pd.read_csv('./clean_datasets/orders_clean.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items     = pd.read_csv('./clean_datasets/items_clean.csv')
products  = pd.read_csv('./clean_datasets/products_clean.csv')
customers = pd.read_csv('./clean_datasets/customers_clean.csv')
payments  = pd.read_csv('./clean_datasets/payments_clean.csv')
reviews   = pd.read_csv('./clean_datasets/reviews_clean.csv')
sellers   = pd.read_csv('./clean_datasets/sellers_clean.csv')
geoloc    = pd.read_csv('./clean_datasets/geoloc_agg.csv')


### dim_customers

In [2]:
dim_customers = customers.merge(
    geoloc[['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']],
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
).drop(columns='geolocation_zip_code_prefix')

dim_customers['customer_zip_code_prefix'] = (
    dim_customers['customer_zip_code_prefix']
    .astype(str)
    .str.zfill(5)
)

dim_customers.to_csv('./powerbi_datasets/dim_customers.csv', index=False)
print(f'dim_customers: {dim_customers.shape}')
assert dim_customers['customer_id'].duplicated().sum() == 0

dim_customers: (99441, 9)


### dim_sellers

In [3]:
dim_sellers = sellers.merge(
    geoloc[['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']],
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
).drop(columns='geolocation_zip_code_prefix')

dim_sellers['seller_zip_code_prefix'] = (
    dim_sellers['seller_zip_code_prefix']
    .astype(str)
    .str.zfill(5)
)

dim_sellers.to_csv('./powerbi_datasets/dim_sellers.csv', index=False)
print(f'dim_sellers: {dim_sellers.shape}')
assert dim_sellers['seller_id'].duplicated().sum() == 0

dim_sellers: (3095, 8)


### dim_products

In [4]:
dim_products = products[~products['is_physically_invalid']].copy()
dim_products = dim_products.drop(columns='is_physically_invalid')

dim_products.to_csv('./powerbi_datasets/dim_products.csv', index=False)
print(f'dim_products: {dim_products.shape}')
assert dim_products['product_id'].duplicated().sum() == 0

dim_products: (32947, 9)


### dim_date

In [5]:
date_range = pd.date_range(start=orders['order_purchase_timestamp'].min(),
    end=orders['order_purchase_timestamp'].max(),
    freq='D')

dim_date = pd.DataFrame({'date': date_range})

dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['week'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['day'] = dim_date['date'].dt.day
dim_date['day_of_week'] = dim_date['date'].dt.dayofweek
dim_date['day_name'] = dim_date['date'].dt.strftime('%A')
dim_date['is_weekend'] = dim_date['day_of_week'] >= 5
dim_date['year_month'] = dim_date['date'].dt.to_period('M').astype(str)
dim_date['date'] = pd.to_datetime(dim_date['date']).dt.normalize()

dim_date.to_csv('./powerbi_datasets/dim_date.csv', index=False)
print(f'dim_date: {dim_date.shape}')
print(f'Date range: {dim_date["date"].min()} → {dim_date["date"].max()}')

dim_date: (773, 11)
Date range: 2016-09-04 00:00:00 → 2018-10-16 00:00:00


### fact_orders

In [13]:
fact_orders = orders[
    (orders['order_status'] == 'delivered') &
    (orders['has_invalid_dates'] == False)
].copy()

fact_orders['delivery_days'] = (
    fact_orders['order_delivered_customer_date'] - fact_orders['order_purchase_timestamp']
).dt.days

fact_orders['days_vs_estimate'] = (
    fact_orders['order_delivered_customer_date'] - fact_orders['order_estimated_delivery_date']
).dt.days

fact_orders['delivered_late'] = fact_orders['days_vs_estimate'] > 0

items_agg = items.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    has_free_shipping=('is_free_shipping', 'any')
).reset_index()

payments_agg = payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', 'first')
).reset_index()

reviews_agg = reviews[['order_id', 'review_score']].drop_duplicates(subset='order_id')

fact_orders = fact_orders.merge(payments_agg, on='order_id', how='left')
fact_orders = fact_orders.merge(items_agg, on='order_id', how='left')
fact_orders = fact_orders.merge(reviews_agg, on='order_id', how='left')

# Extract hour before normalizing
fact_orders['purchase_hour'] = fact_orders['order_purchase_timestamp'].dt.hour

fact_orders['order_purchase_timestamp'] = fact_orders['order_purchase_timestamp'].dt.normalize()

fact_orders = fact_orders[[
    'order_id', 'customer_id',
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'purchase_hour',
    'delivery_days', 'days_vs_estimate', 'delivered_late',
    'total_items', 'total_price', 'total_freight', 'has_free_shipping',
    'total_payment_value', 'payment_installments', 'payment_type',
    'review_score'
]]

fact_orders = fact_orders[fact_orders['payment_type'].notna() & 
                          (fact_orders['payment_type'] != '')
                        ]

print(f"fact_orders after payment_type filter: {len(fact_orders):,} rows")

fact_orders.to_csv('./powerbi_datasets/fact_orders.csv', index=False)
print(f'fact_orders: {fact_orders.shape}')
assert fact_orders['order_id'].duplicated().sum() == 0
print(f'Null review_score: {fact_orders["review_score"].isna().sum()}')

fact_orders after payment_type filter: 95,103 rows
fact_orders: (95103, 19)
Null review_score: 1122


### fact_items

In [9]:
fact_items = items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

# Keep only orders that exist in fact_orders (delivered + valid dates)
valid_orders = set(fact_orders['order_id'])
fact_items = fact_items[fact_items['order_id'].isin(valid_orders)].copy()

fact_items = fact_items[[
    'order_id', 'order_item_id', 'product_id',
    'seller_id','price', 'freight_value', 'is_free_shipping'
]]

fact_items.to_csv('./powerbi_datasets/fact_items.csv', index=False)
print(f'fact_items: {fact_items.shape}')
assert fact_items.duplicated(subset=['order_id', 'order_item_id']).sum() == 0

fact_items: (108605, 7)
